In [1]:
cnt = 0
goals  = []
opss = []
voltss = []
# with open('sample') as file:
with open('input') as file:
    while line := file.readline():
        # print(line)
        goals.append( line[1:line.index(']')])
        opss.append(line[line.index(']')+2:line.index('{')-1])
        voltss.append(line[line.index('{')+1:line.index('}')])
        cnt +=1

In [2]:
import numpy as np
import sympy as sp
import itertools
buttons = ([np.array(eval('['+x[1:-1]+']')) for x in opss[0].split(' ')])
goal = np.array([int(x) for x in voltss[0].split(',')])

In [5]:
import numpy as np
import pulp as pl

total_count = 0

for idx, buttons in enumerate(opss):
    start = np.array([int(x) for x in voltss[idx].split(',')], dtype=int)
    goal  = np.array([0 for _ in range(len(start))], dtype=int)
    buttons = eval(buttons.replace(')', ',)').replace(') (', '), ('))
    
    B = np.zeros(shape=(len(buttons),len(start))) 
    for b_idx, vals in enumerate(buttons):
        for v in vals:
            B[b_idx][v] = 1


    demand = start - goal
    m, n = B.shape

    prob = pl.LpProblem("min_button_presses", pl.LpMinimize)
    x = [pl.LpVariable(f"x_{j}", lowBound=0, cat="Integer") for j in range(m)]

    prob += pl.lpSum(x)
    for i in range(n):
        prob += pl.lpSum(int(B[j, i]) * x[j] for j in range(m)) == int(demand[i])

    status = prob.solve(pl.PULP_CBC_CMD(msg=False))
    if pl.LpStatus[status] != "Optimal":
        print("No optimal solution:", pl.LpStatus[status])
    else:
        x_sol = [int(pl.value(xj)) for xj in x]
        total_count += sum(x_sol)

total_count



17576

In [3]:
def create_mat(buttons, goals):
    mat = np.zeros(shape=(len(goals), len(buttons)+1))

    # each button is a column 
    for button_idx, button in enumerate(buttons):
        for col_idx in button:
            mat[col_idx][button_idx] = 1
    
    # the last column are the goal values
    for goal_idx, goal in enumerate(goals):
        mat[goal_idx][-1] = goal
    
    return mat


m = (create_mat(buttons, goal))
print(m)

[[0. 0. 0. 0. 1. 1. 3.]
 [0. 1. 0. 0. 0. 1. 5.]
 [0. 0. 1. 1. 1. 0. 4.]
 [1. 1. 0. 1. 0. 0. 7.]]


In [4]:
def scale_row(mat, row_idx, scale_val):
    mat[row_idx] *= scale_val
    return mat

m = scale_row(m, 2, 200)
m

array([[  0.,   0.,   0.,   0.,   1.,   1.,   3.],
       [  0.,   1.,   0.,   0.,   0.,   1.,   5.],
       [  0.,   0., 200., 200., 200.,   0., 800.],
       [  1.,   1.,   0.,   1.,   0.,   0.,   7.]])

In [6]:
lines =  open('input').read().strip().splitlines()
total_count = 0
for line in lines:
    line = line.split()
    goal = int(line[0][1:-1].replace('.','0').replace('#','1')[::-1],2)
    ops = [sum(1 << int(val) for val in eval('['+button[1:-1]+']')) for button in line[1:-1]]

    s = {0}
    this_count = 0
    while goal not in s:
        s |= {op^val for op in ops for val in s}
        this_count +=1
    total_count += this_count

print(total_count)
                

457


In [7]:
# import numpy as np

# def solver(goal_str, ops_str):
#     start = [-1] * len(goal_str)
#     # goal = [-1, 1, 1, -1]
#     goal = [-1 if c=='.' else 1 for c in goal_str]
#     # s = "(3) (1,3) (2) (2,3) (0,2) (0,1)" 
#     s = ops_str
#     s = s.replace('(','').replace(')','').split(' ')
#     s = [list(map(int,x.split(','))) for x in s]
#     ops = []
#     for ss in s:
#         r = [1]*len(goal_str)
#         for idx in ss:
#             r[idx] *= -1
#         ops.append(np.array(r))

#     print()
#     print()
#     print()
#     print(start)
#     print(goal, goal_str)
#     print(ops, ops_str)

#     if start == goal:
#         flag = False
#         print("!!!")
#         return 0
#     goal = np.array(goal)
#     start = np.array(start)


#     q = [(start, "")]
    
#     ops.reverse()

#     flag = True
#     while flag and len(q) > 0:
#         key = q.pop(0)

#         for idx,op in enumerate(ops):
#             r = [x*y for x,y in zip(op, key[0])]
#             r = op * key[0]
#             rs = key[1] + str(idx)
#             # print(rs,r, list(zip(op,key[0])))


#             if r == goal:
#                 print(r,rs)
#                 print(len(rs))
#                 flag = False
#                 return len(rs)

#             else:
#                 q.append((r,rs))



# # solver(".##.", "(3) (1,3) (2) (2,3) (0,2) (0,1)")
# ts = 0
# for c in range(cnt):
#     # print(goals[c], opss[c])
#     ts += solver(goals[c], opss[c])
# ts
